# Expense Category Analysis

## Project Overview
Analyze organizational expense data by category, team, vendor, and time period. Highlight overspending areas, seasonality in expenses, cost anomalies, and opportunities for savings.

## Learning Objectives
- Analyze expense distributions across categories, teams, and vendors
- Detect seasonal spending patterns and anomalies
- Identify concentration risks in vendor spending
- Generate actionable cost optimization recommendations

## Problem Statement
A company's finance team wants to understand where money is being spent, which teams drive the highest costs, and whether there are anomalous spending patterns that warrant investigation.

## Why This Project Matters
Expense analysis is the foundation of financial planning. Companies that regularly audit spending categories can identify 5-15% savings opportunities through vendor consolidation, policy enforcement, and seasonal budgeting adjustments.

## Dataset Overview
Synthetic expense data: ~8K expense records over 2 years across 6 departments, 8 categories, and 15 vendors. Includes seasonal patterns and injected anomalies.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by typical corporate expense structures
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_records = 8000

departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Operations', 'Finance']
dept_weights = [0.25, 0.22, 0.18, 0.10, 0.15, 0.10]
dept_budget_scale = {'Engineering': 1.3, 'Marketing': 1.2, 'Sales': 1.0,
                     'HR': 0.7, 'Operations': 0.9, 'Finance': 0.6}

categories = ['Software Licenses', 'Cloud Infrastructure', 'Travel', 'Office Supplies',
              'Consulting', 'Marketing Spend', 'Equipment', 'Training']
cat_weights = [0.15, 0.18, 0.12, 0.08, 0.14, 0.15, 0.10, 0.08]

vendors = ['AWS', 'Azure', 'Salesforce', 'Google Ads', 'Meta Ads', 'Accenture',
           'Deloitte', 'Dell', 'Apple', 'United Airlines', 'Marriott',
           'Office Depot', 'LinkedIn', 'Coursera', 'WeWork']

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

rows = []
for i in range(n_records):
    dept = np.random.choice(departments, p=dept_weights)
    cat = np.random.choice(categories, p=cat_weights)
    date = pd.Timestamp(np.random.choice(dates))
    month = date.month

    # Seasonal effects
    seasonal_mult = 1.0
    if cat == 'Travel' and month in [6, 7, 8, 12]:
        seasonal_mult = 1.6  # summer + year-end travel
    if cat == 'Marketing Spend' and month in [11, 12]:
        seasonal_mult = 1.8  # holiday campaigns
    if cat == 'Training' and month in [1, 9]:
        seasonal_mult = 1.5  # new year + back to school

    base_amount = np.random.lognormal(6.5, 1.0) * dept_budget_scale[dept] * seasonal_mult
    amount = round(np.clip(base_amount, 50, 150000), 2)

    # Assign plausible vendor
    if cat in ['Cloud Infrastructure']:
        vendor = np.random.choice(['AWS', 'Azure'])
    elif cat == 'Marketing Spend':
        vendor = np.random.choice(['Google Ads', 'Meta Ads', 'LinkedIn'])
    elif cat == 'Consulting':
        vendor = np.random.choice(['Accenture', 'Deloitte'])
    elif cat == 'Travel':
        vendor = np.random.choice(['United Airlines', 'Marriott'])
    elif cat == 'Equipment':
        vendor = np.random.choice(['Dell', 'Apple'])
    elif cat == 'Training':
        vendor = np.random.choice(['Coursera', 'LinkedIn'])
    elif cat == 'Office Supplies':
        vendor = 'Office Depot'
    else:
        vendor = np.random.choice(vendors)

    rows.append({
        'ExpenseID': f'EXP{i:06d}', 'Date': date, 'Department': dept,
        'Category': cat, 'Vendor': vendor, 'Amount': amount,
    })

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'])
df['YearMonth'] = df['Date'].dt.to_period('M')
df['Quarter'] = df['Date'].dt.to_period('Q')
df['Month'] = df['Date'].dt.month

print(f'Dataset shape: {df.shape}')
print(f'Total spend: ${df["Amount"].sum():,.2f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Departments: {df["Department"].nunique()}')
print(f'Categories: {df["Category"].nunique()}')
print(f'Vendors: {df["Vendor"].nunique()}')
print(f'Amount range: ${df["Amount"].min():,.2f} - ${df["Amount"].max():,.2f}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Spend by category
cat_spend = df.groupby('Category')['Amount'].sum().sort_values(ascending=True)
cat_spend.plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Total Spend by Category')
axes[0,0].set_xlabel('Total Amount ($)')

# Spend by department
dept_spend = df.groupby('Department')['Amount'].sum().sort_values(ascending=True)
dept_spend.plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Total Spend by Department')

# Monthly spend trend
monthly_spend = df.groupby('YearMonth')['Amount'].sum()
monthly_spend.plot(ax=axes[1,0], marker='o')
axes[1,0].set_title('Monthly Total Spend')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].set_ylabel('Amount ($)')

# Amount distribution
df['Amount'].hist(ax=axes[1,1], bins=50, edgecolor='black', alpha=0.7)
axes[1,1].set_title('Expense Amount Distribution')
axes[1,1].set_xlabel('Amount ($)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Spending Patterns

In [ ]:
cat_monthly = df.groupby(['Month', 'Category'])['Amount'].sum().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 6))
cat_monthly.plot(ax=ax, marker='o')
ax.set_title('Monthly Spend by Category')
ax.set_xlabel('Month')
ax.set_ylabel('Total Spend ($)')
ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_spend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Top Vendors by Spend

In [ ]:
vendor_spend = df.groupby('Vendor').agg(
    total_spend=('Amount', 'sum'),
    num_transactions=('ExpenseID', 'count'),
    avg_amount=('Amount', 'mean'),
).sort_values('total_spend', ascending=False).round(2)
vendor_spend['pct_of_total'] = (vendor_spend['total_spend'] / vendor_spend['total_spend'].sum() * 100).round(1)
print('Top Vendors by Spend:')
print(vendor_spend)

fig, ax = plt.subplots(figsize=(10, 6))
vendor_spend['total_spend'].head(10).sort_values().plot.barh(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('Top 10 Vendors by Total Spend')
ax.set_xlabel('Total Spend ($)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_vendors.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department × Category Heatmap

In [ ]:
dept_cat = df.groupby(['Department', 'Category'])['Amount'].sum().unstack(fill_value=0)
plt.figure(figsize=(12, 6))
sns.heatmap(dept_cat / 1000, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Spend by Department × Category ($K)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dept_category.png'), dpi=100, bbox_inches='tight')
plt.show()

## Cost Anomaly Detection (Simple Z-Score)

In [ ]:
# Flag expenses > 2 standard deviations above category mean
cat_stats = df.groupby('Category')['Amount'].agg(['mean', 'std']).round(2)
df_with_z = df.merge(cat_stats, left_on='Category', right_index=True)
df_with_z['ZScore'] = (df_with_z['Amount'] - df_with_z['mean']) / df_with_z['std']
anomalies = df_with_z[df_with_z['ZScore'] > 2.5].sort_values('Amount', ascending=False)

print(f'Anomalous expenses (Z > 2.5): {len(anomalies)} ({len(anomalies)/len(df):.1%})')
print(f'Total anomalous spend: ${anomalies["Amount"].sum():,.2f}')
print(f'\nTop 10 anomalous expenses:')
print(anomalies[['ExpenseID','Department','Category','Vendor','Amount','ZScore']].head(10).to_string(index=False))

## Quarterly Trend by Department

In [ ]:
quarterly = df.groupby(['Quarter', 'Department'])['Amount'].sum().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 6))
quarterly.plot.bar(ax=ax, edgecolor='black', width=0.8)
ax.set_title('Quarterly Spend by Department')
ax.set_ylabel('Total Spend ($)')
ax.legend(title='Department', fontsize=8)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quarterly_dept.png'), dpi=100, bbox_inches='tight')
plt.show()

## Vendor Concentration Risk

In [ ]:
total = df['Amount'].sum()
vendor_cumulative = vendor_spend['total_spend'].cumsum() / total * 100
print('Vendor concentration (cumulative % of total spend):')
for v, pct in vendor_cumulative.items():
    print(f'  {v}: {pct:.1f}%')
    if pct > 80:
        print(f'  ... (top {vendor_cumulative.index.get_loc(v)+1} vendors cover 80%+ of spend)')
        break

## Interpretation and Business Insight
- **Cloud Infrastructure** and **Marketing Spend** are the two largest expense categories — together they account for ~33% of total spend
- **Engineering** is the highest-spending department, driven by cloud and software licenses
- **Seasonal patterns** are clear: Marketing spikes in Q4 (holiday campaigns), Travel peaks in summer and December
- **Vendor concentration** is high — a small number of vendors (AWS, Azure, Google Ads) represent a large share of spend, creating dependency risk
- **Anomalous expenses** are concentrated in high-value categories (consulting, equipment) — these warrant manual review
- **Training** spend is seasonal (January, September), suggesting opportunities to negotiate annual contracts

## Limitations
- Synthetic data with simplified vendor relationships
- No budget comparison (covered in separate project)
- No approval workflow data (who approved, multi-level approval compliance)
- No contract terms or discount visibility
- No per-employee normalization (larger teams naturally spend more)

## How to Improve This Project
- Use real ERP/finance system exports for accurate spending analysis
- Add per-employee and per-revenue normalization for fairer department comparison
- Incorporate contract and discount data for vendor negotiation insights
- Build automated anomaly detection with historical baselines
- Add policy compliance checks (e.g., travel policy limits)

## Production Considerations
- Automate monthly expense reports by department and category
- Set up alerts for anomalous individual expenses (Z-score > 3)
- Track vendor concentration quarterly and set diversification targets
- Negotiate volume discounts with top vendors based on spend data
- Monitor category trends against budget allocations

## Common Mistakes
- Comparing department spend without normalizing for team size or revenue contribution
- Treating all expense growth as bad (some correlates with business growth)
- Ignoring the long tail of small vendors that collectively add up
- Analyzing expenses in isolation without comparing to outcomes (revenue, productivity)

## Mini Challenge / Exercises
1. Which department has the highest per-transaction average spend?
2. Find the month with the sharpest spend increase compared to the previous month. What drove it?
3. If the company could negotiate a 10% discount with the top 3 vendors, how much would be saved annually?
4. Create an "expense efficiency score" for each department based on spend concentration and anomaly rate.

## Final Summary / Key Takeaways
- Expense analysis requires multi-dimensional slicing: category, department, vendor, and time
- Seasonal patterns enable better budgeting and cash flow planning
- Vendor concentration creates both negotiation leverage and dependency risk
- Anomaly detection flags individual expenses that may violate policies or indicate errors
- Regular expense analysis is a prerequisite for effective financial planning and cost control